# Provision Fabric Workspace, Lakehouse, and Warehouse — Insurance Demo Jump-Start

**Purpose**: Provision the core Fabric data stores for the
`FSI-Fabric-Medallion-Architecture-Insurance-Demo` solution. No Git sync,
no item-definition push.

**What this notebook does** (idempotent — safe to re-run):
1. Creates or reuses the target Fabric **workspace**.
2. Creates the **`LH_Insurance` Lakehouse** and captures its **SQL analytics
   endpoint** connection string.
3. Creates the **`WH_Insurance` Warehouse** and captures its **SQL endpoint**
   connection string.

**After this notebook**, complete the data load manually (see the repository
README for detailed steps):
1. Upload the reference data from the GitHub repo into `LH_Insurance`.
2. Run `02_insurance_bronze_to_silver` to build the Silver Delta tables.
3. Run the Warehouse stored procedure `gold.usp_Load_Silver_to_Gold` to load
   the Gold KPI tables.

**Prerequisites**:
- Run this notebook **inside a Fabric workspace** (any bootstrap workspace if
  you set `TARGET_WORKSPACE_DISPLAY_NAME`, or the target workspace itself if
  you set `WORKSPACE_ID`).
- The target workspace must have a **Fabric capacity** assigned.
- You must have **Contributor** or **Admin** on the target workspace.


## 0. Configuration

In [ ]:
# ============================================================
# CONFIGURATION — update these values for your environment
# ============================================================

# --- Target workspace -------------------------------------------------------
# Option A (run inside the target workspace): paste its WORKSPACE_ID and leave
#   TARGET_WORKSPACE_DISPLAY_NAME as-is; the notebook will use WORKSPACE_ID.
# Option B (create/reuse by name from any bootstrap workspace): leave
#   WORKSPACE_ID blank and set CAPACITY_ID; the notebook creates or reuses the
#   workspace named TARGET_WORKSPACE_DISPLAY_NAME.
WORKSPACE_ID = ""  # e.g. "11111111-2222-3333-4444-555555555555"

TARGET_WORKSPACE_DISPLAY_NAME = "FSI-Fabric-Medallion-Architecture-Insurance-Demo"
TARGET_WORKSPACE_DESCRIPTION = (
    "Insurance demo workspace for the "
    "FSI-Fabric-Medallion-Architecture-Insurance-Demo solution."
)

# Required only when WORKSPACE_ID is blank (create/reuse by name).
CAPACITY_ID = "5d90d327-f4a8-4efa-ab8e-665d090a49ab"
DOMAIN_ID = ""

# --- Data stores ------------------------------------------------------------
LAKEHOUSE_NAME = "LH_Insurance"
WAREHOUSE_NAME = "WH_Insurance"

print("Configuration loaded.")
print(f"  Workspace name : {TARGET_WORKSPACE_DISPLAY_NAME}")
print(f"  Workspace id   : {WORKSPACE_ID or '(create/reuse by name)'}")
print(f"  Lakehouse      : {LAKEHOUSE_NAME}")
print(f"  Warehouse      : {WAREHOUSE_NAME}")


## 1. Authentication and Helper Functions

In [ ]:
import json
import time
from typing import Any, Dict, List, Optional

import requests

FABRIC_API_BASE = "https://api.fabric.microsoft.com/v1"


def _notebookutils():
    """Return the Fabric notebookutils module (only available inside Fabric)."""
    try:
        return notebookutils  # type: ignore[name-defined]
    except NameError:
        try:
            import notebookutils as _nbu  # type: ignore
            return _nbu
        except ImportError as exc:
            raise RuntimeError(
                "This notebook must run in Microsoft Fabric, where "
                "notebookutils is available."
            ) from exc


def fabric_token() -> str:
    return _notebookutils().credentials.getToken("pbi")


def auth_headers() -> Dict[str, str]:
    return {
        "Authorization": f"Bearer {fabric_token()}",
        "Content-Type": "application/json",
    }


def fabric_request(method: str, url: str, *, json_body: Optional[Dict[str, Any]] = None,
                   timeout: int = 120) -> requests.Response:
    """Call the Fabric REST API, transparently retrying on 429 throttling."""
    full_url = url if url.startswith("https://") else f"{FABRIC_API_BASE}{url}"
    while True:
        resp = requests.request(method, full_url, headers=auth_headers(),
                                json=json_body, timeout=timeout)
        if resp.status_code == 429:
            retry_after = int(resp.headers.get("Retry-After", "30"))
            print(f"  Throttled by Fabric API. Retrying in {retry_after}s...")
            time.sleep(retry_after)
            continue
        return resp


def response_json(resp: requests.Response) -> Dict[str, Any]:
    return resp.json() if resp.text else {}


def wait_for_lro(resp: requests.Response, *, timeout_minutes: int = 30) -> Dict[str, Any]:
    """Poll a Fabric long-running operation (HTTP 202) until it finishes."""
    if resp.status_code != 202:
        return response_json(resp)

    op_url = resp.headers.get("Location", "")
    operation_id = resp.headers.get("x-ms-operation-id")
    if not op_url and operation_id:
        op_url = f"{FABRIC_API_BASE}/operations/{operation_id}"
    if not op_url:
        return response_json(resp)

    deadline = time.time() + timeout_minutes * 60
    while time.time() < deadline:
        retry_after = int(resp.headers.get("Retry-After", "20"))
        time.sleep(retry_after)
        state_resp = fabric_request("GET", op_url)
        state = response_json(state_resp)
        status = state.get("status")
        if status == "Succeeded":
            result_url = op_url.rstrip("/") + "/result"
            result_resp = fabric_request("GET", result_url)
            if result_resp.status_code == 200 and result_resp.text:
                return response_json(result_resp)
            return state
        if status == "Failed":
            raise RuntimeError(f"Fabric operation failed: {json.dumps(state)[:500]}")
        resp = state_resp  # refresh Retry-After for next loop

    raise TimeoutError(f"Fabric operation did not finish within {timeout_minutes} min.")


def create_item_idempotent(workspace_id: str, display_name: str, item_type: str) -> Dict[str, Any]:
    """Create a Fabric item; treat HTTP 409 (already exists) as success."""
    resp = fabric_request(
        "POST",
        f"/workspaces/{workspace_id}/items",
        json_body={"displayName": display_name, "type": item_type},
    )
    if resp.status_code == 409:
        return {"status": "already_exists"}
    if resp.status_code in (200, 201):
        return response_json(resp) or {"status": "created"}
    if resp.status_code == 202:
        return wait_for_lro(resp) or {"status": "created"}
    raise RuntimeError(
        f"Failed to create {item_type} '{display_name}': "
        f"HTTP {resp.status_code} - {resp.text[:400]}"
    )


def list_items(workspace_id: str, item_type: Optional[str] = None) -> List[Dict[str, Any]]:
    items: List[Dict[str, Any]] = []
    path = f"/workspaces/{workspace_id}/items"
    if item_type:
        path += f"?type={item_type}"
    while path:
        resp = fabric_request("GET", path)
        resp.raise_for_status()
        payload = response_json(resp)
        items.extend(payload.get("value", []))
        token = payload.get("continuationToken")
        if not token:
            break
        sep = "&" if "?" in path.split("/workspaces", 1)[-1] else "?"
        base = f"/workspaces/{workspace_id}/items"
        path = f"{base}?type={item_type}&continuationToken={token}" if item_type \
            else f"{base}?continuationToken={token}"
    return items


def find_item_id(workspace_id: str, display_name: str, item_type: str) -> Optional[str]:
    for item in list_items(workspace_id, item_type):
        if item.get("displayName") == display_name:
            return item.get("id")
    return None


print("Helper functions loaded.")


## 2. Resolve or Create the Target Workspace

If `WORKSPACE_ID` is set, that workspace is used directly. Otherwise the
notebook creates (or reuses) a workspace named `TARGET_WORKSPACE_DISPLAY_NAME`
and assigns `CAPACITY_ID`.

In [ ]:
def list_workspaces() -> List[Dict[str, Any]]:
    workspaces: List[Dict[str, Any]] = []
    path = "/workspaces"
    while path:
        resp = fabric_request("GET", path)
        resp.raise_for_status()
        payload = response_json(resp)
        workspaces.extend(payload.get("value", []))
        token = payload.get("continuationToken")
        path = f"/workspaces?continuationToken={token}" if token else ""
    return workspaces


def resolve_workspace() -> str:
    if WORKSPACE_ID:
        info = fabric_request("GET", f"/workspaces/{WORKSPACE_ID}")
        info.raise_for_status()
        name = response_json(info).get("displayName", "Unknown")
        print(f"Using existing workspace by id: {name} ({WORKSPACE_ID})")
        return WORKSPACE_ID

    matches = [w for w in list_workspaces()
               if w.get("displayName") == TARGET_WORKSPACE_DISPLAY_NAME]
    if len(matches) > 1:
        raise RuntimeError(
            f"Multiple workspaces named {TARGET_WORKSPACE_DISPLAY_NAME!r} found; "
            "set WORKSPACE_ID explicitly."
        )
    if matches:
        ws = matches[0]
        print(f"Reusing workspace: {ws['displayName']} ({ws['id']})")
        return ws["id"]

    body: Dict[str, Any] = {
        "displayName": TARGET_WORKSPACE_DISPLAY_NAME,
        "description": TARGET_WORKSPACE_DESCRIPTION,
    }
    if CAPACITY_ID:
        body["capacityId"] = CAPACITY_ID
    if DOMAIN_ID:
        body["domainId"] = DOMAIN_ID

    resp = fabric_request("POST", "/workspaces", json_body=body)
    if resp.status_code not in (200, 201):
        raise RuntimeError(
            f"Failed to create workspace: HTTP {resp.status_code} - {resp.text[:400]}"
        )
    ws = response_json(resp)
    print(f"Created workspace: {ws['displayName']} ({ws['id']})")
    return ws["id"]


workspace_id = resolve_workspace()
print(f"\nActive workspace id: {workspace_id}")


## 3. Create the Lakehouse (LH_Insurance) and Capture Its SQL Endpoint

In [ ]:
print("=" * 60)
print("STEP 3: Create Lakehouse")
print("=" * 60)

result = create_item_idempotent(workspace_id, LAKEHOUSE_NAME, "Lakehouse")
if result.get("status") == "already_exists":
    print(f"  {LAKEHOUSE_NAME} already exists — skipped.")
else:
    print(f"  Created {LAKEHOUSE_NAME}.")

# Retrieve the id (with a short retry for propagation).
lakehouse_id = None
for attempt in range(6):
    lakehouse_id = find_item_id(workspace_id, LAKEHOUSE_NAME, "Lakehouse")
    if lakehouse_id:
        break
    time.sleep(3)

if not lakehouse_id:
    raise RuntimeError(f"Could not resolve id for lakehouse {LAKEHOUSE_NAME}.")
print(f"  {LAKEHOUSE_NAME} -> {lakehouse_id}")

# Capture the Lakehouse SQL analytics endpoint. It is provisioned
# asynchronously, so poll briefly until the connection string is ready.
lakehouse_sql_endpoint = None
lakehouse_sql_endpoint_id = None
for attempt in range(10):
    details = fabric_request(
        "GET", f"/workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    if details.status_code == 200:
        sql_props = response_json(details).get("properties", {}).get(
            "sqlEndpointProperties", {})
        status = sql_props.get("provisioningStatus")
        lakehouse_sql_endpoint = sql_props.get("connectionString")
        lakehouse_sql_endpoint_id = sql_props.get("id")
        if lakehouse_sql_endpoint and status in (None, "Success"):
            break
        print(f"    SQL endpoint provisioning: {status or 'pending'} "
              f"({attempt + 1}/10)...")
    time.sleep(6)

if lakehouse_sql_endpoint:
    print(f"  {LAKEHOUSE_NAME} SQL endpoint: {lakehouse_sql_endpoint}")
else:
    print(f"  {LAKEHOUSE_NAME} SQL endpoint not ready yet; check the Lakehouse "
          "SQL analytics endpoint in the portal before running the stored proc.")


## 4. Create the Warehouse (WH_Insurance) and Capture Its SQL Endpoint

In [ ]:
print("=" * 60)
print("STEP 4: Create Warehouse")
print("=" * 60)

result = create_item_idempotent(workspace_id, WAREHOUSE_NAME, "Warehouse")
if result.get("status") == "already_exists":
    print(f"  {WAREHOUSE_NAME} already exists — skipped.")
else:
    print(f"  Created {WAREHOUSE_NAME}.")

warehouse_id = None
for attempt in range(8):
    warehouse_id = find_item_id(workspace_id, WAREHOUSE_NAME, "Warehouse")
    if warehouse_id:
        break
    print(f"    Waiting for warehouse to propagate ({attempt + 1}/8)...")
    time.sleep(4)

if not warehouse_id:
    raise RuntimeError(f"Could not resolve id for warehouse {WAREHOUSE_NAME}.")
print(f"  {WAREHOUSE_NAME} -> {warehouse_id}")

# Capture the Warehouse T-SQL connection string (SQL endpoint). Poll briefly
# because it can lag item creation. Use it to connect (SSMS, Azure Data
# Studio, or the Fabric SQL editor) and run gold.usp_Load_Silver_to_Gold.
warehouse_sql_endpoint = None
for attempt in range(10):
    details = fabric_request(
        "GET", f"/workspaces/{workspace_id}/warehouses/{warehouse_id}")
    if details.status_code == 200:
        props = response_json(details).get("properties", {})
        warehouse_sql_endpoint = props.get("connectionString") or props.get("connectionInfo")
        if warehouse_sql_endpoint:
            break
    print(f"    Waiting for warehouse SQL endpoint ({attempt + 1}/10)...")
    time.sleep(6)

if warehouse_sql_endpoint:
    print(f"  {WAREHOUSE_NAME} SQL endpoint: {warehouse_sql_endpoint}")
else:
    print(f"  {WAREHOUSE_NAME} SQL endpoint not ready yet; check the Warehouse "
          "connection string in the portal.")


## 5. Provisioning Summary and Manual Next Steps

In [ ]:
print("=" * 60)
print("PROVISIONING COMPLETE")
print("=" * 60)
print(f"""
  Workspace : {TARGET_WORKSPACE_DISPLAY_NAME}
              {workspace_id}
  Lakehouse : {LAKEHOUSE_NAME} ({lakehouse_id})
              SQL endpoint: {lakehouse_sql_endpoint or '(check portal)'}
  Warehouse : {WAREHOUSE_NAME} ({warehouse_id})
              SQL endpoint: {warehouse_sql_endpoint or '(check portal)'}

  This notebook provisions data stores only. Complete the data load manually:
    1. Upload the reference data from the GitHub repo into {LAKEHOUSE_NAME}
       (run 01_upload_reference_data_to_lakehouse, or upload the
       Reference_Data files into Files/Bronze_Raw_Data).
    2. Run 02_insurance_bronze_to_silver to build the Silver Delta tables.
    3. Deploy and run the Warehouse stored procedure
       gold.usp_Load_Silver_to_Gold to load the Gold KPI tables.

  See the repository README for detailed manual steps.

  Workspace URL:
    https://app.fabric.microsoft.com/groups/{workspace_id}
""")
